# ColdSite-DTI — the 24-run training grid (Colab, T4)

**2 datasets x 4 splits x 3 training seeds = 24 runs**, regression, ColdSite-DTI only.
This is STATUS.md item 6, the critical path. It is *not* the 36-run DAVIS binary
baseline grid — that one is `colab_davis_grid.ipynb` and trains three other models.

| | |
|---|---|
| Runtime needed | **T4 GPU** (Runtime -> Change runtime type -> T4 GPU) |
| DAVIS, 12 cells | ~3-7 GPU-hours |
| KIBA, 12 cells | ~12-27 GPU-hours (KIBA has 118k pairs against DAVIS's 30k) |
| Sessions | expect several — Colab disconnects |
| Resumable | yes, and safely: see below |

**A TPU runtime will not work.** `torch.cuda.is_available()` is `False` there, so the
trainer silently falls back to 2 vCPUs. Cell 1 stops if this is not a GPU.

Everything writes to Google Drive, so a disconnect costs one cell, not one session.
Re-run this notebook from the top after any drop: finished cells are skipped, and an
**interrupted cell is retrained rather than skipped** (commit `d9a03c1`) — before that
fix a half-trained cell was banked as a finished result with no accuracy.


## 1. Check the runtime


In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No CUDA device. Runtime -> Change runtime type -> T4 GPU. '
    'A TPU runtime falls back to CPU and this grid would take weeks.'
)

name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU     : {name}')
print(f'memory  : {total_gb:.1f} GB')
print(f'torch   : {torch.__version__}')

# ColdSite-DTI, forward + backward, 1000-residue protein:
#   batch 64 -> 8.7 GB   batch 16 -> ~2.5 GB
BATCH = 64
if total_gb < 14:
    BATCH = 16
    print(f'\nUnder 14 GB — dropping to batch {BATCH}.')
print('batch   :', BATCH)


## 2. Mount Drive

Checkpoints, splits and the DeepDTA source files all live here so a runtime reset
does not mean re-downloading and re-building before training can resume.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Separate from the binary baseline grid's folder on purpose: that grid writes
# binary-task checkpoints for three models, this one writes regression
# checkpoints for ColdSite-DTI. Sharing a directory would mix two tasks.
DRIVE_RESULTS = '/content/drive/MyDrive/coldsite-grid24-results'
# Shared with the other notebook — same splits, same source files, built once.
DRIVE_SPLITS = '/content/drive/MyDrive/coldsite-splits'
DRIVE_DEEPDTA_DATA = '/content/drive/MyDrive/coldsite-deepdta-data'

for path in (DRIVE_RESULTS, DRIVE_SPLITS, DRIVE_DEEPDTA_DATA):
    os.makedirs(path, exist_ok=True)

print('results     ->', DRIVE_RESULTS)
print('splits      ->', DRIVE_SPLITS)
print('deepdta src ->', DRIVE_DEEPDTA_DATA)
print('existing checkpoints:',
      len([f for f in os.listdir(DRIVE_RESULTS) if f.endswith('.pt')]), '/ 24')


## 3. Clone the repo and install

**This clones the fork at `main`.** Not udayraj's repo, and not `project-completion`,
which is 24 commits behind and lacks the finished control arm.


In [ ]:
REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
BRANCH = 'main'

%cd /content
if not os.path.exists('/content/ColdSite-DTI_New'):
    !git clone --branch {BRANCH} {REPO}
%cd /content/ColdSite-DTI_New
!git checkout {BRANCH} && git pull origin {BRANCH}

!pip install -q tabulate subword-nmt pytest


def ensure_symlink(local_path, drive_path):
    """Make local_path a symlink into Drive. Idempotent."""
    if os.path.islink(local_path):
        return
    if os.path.exists(local_path):
        !rm -rf {local_path}
    parent = os.path.dirname(local_path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    os.symlink(drive_path, local_path)


ensure_symlink('results', DRIVE_RESULTS)
ensure_symlink('data/splits', DRIVE_SPLITS)
ensure_symlink('src/data/baselines/deepdta/data', DRIVE_DEEPDTA_DATA)

print('results      ->', os.path.realpath('results'))
print('data/splits  ->', os.path.realpath('data/splits'))
print('deepdta data ->', os.path.realpath('src/data/baselines/deepdta/data'))
!git log --oneline -1


## 4. Sanity-check the checkout

577 tests in about 15 seconds. Cheap insurance before spending GPU hours.


In [ ]:
!python -m pytest tests/ -q 2>&1 | tail -3


## 5. Fetch the DAVIS/KIBA source files

Gitignored (they belong to DeepDTA), so they come from the original repo. Skipped
entirely if Drive already has them from a previous session.


In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'

for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}

!python -m src.data.load_data


Expected output, exactly:

```
davis: 30056 measured pairs, 68 unique drugs, 442 unique targets, Y range [5.000, 10.796]
kiba: 118254 measured pairs, 2111 unique drugs, 229 unique targets, Y range [0.000, 17.200]
```

If the counts differ, **stop** — the splits would be built on the wrong data.


## 6. Build the splits and preflight

Deterministic and takes ~20 s, so it is cheap to re-run even though it lives in Drive.
All six leakage checks must say OK and preflight must end `ready_to_launch: True`.


In [ ]:
!python -m src.data.build_splits
!python -m src.model.run_grid --preflight 2>&1 | tail -5


Expected for DAVIS (verified locally):

```
random       train=21039  valid=3006  test=6011
cold_drug    train=21658  valid=2652  test=5746
cold_target  train=21080  valid=2992  test=5984
cold_pair    train=15190  valid=264   test=1144
```

`cold_pair` validation really is only 264 rows — inherent to requiring both drug and
target to be unseen. Early stopping on it is noisy across seeds; known, not a bug.


## 7. Train

DAVIS first — it is a quarter of KIBA's size, so a shape error surfaces in minutes
rather than hours. `run_grid` also validates the first cell end to end before
launching the rest.

Both cells are resumable. After a disconnect, re-run the notebook from the top and
then the cell you were on.


### 7a. DAVIS — 12 cells, ~3-7 h


In [ ]:
!python -m src.model.run_grid --datasets davis --batch-size {BATCH}


**Check before continuing.** DAVIS random CI should land around 0.85-0.90, and
cold_pair clearly lower. A CI at 0.99 means a labelling or metric bug, not a good
model — stop and investigate rather than spending 20 more hours on it.


### 7b. KIBA — 12 cells, ~12-27 h

The long one. Start it when you can leave the tab open.


In [ ]:
!python -m src.model.run_grid --datasets kiba --batch-size {BATCH}


## 8. What landed


In [ ]:
import glob

ckpt = glob.glob('results/*regression*.pt')
res = glob.glob('results/*regression*_results.json')
print('checkpoints :', len(ckpt), '/ 24')
print('run results :', len(res), '/ 24')

# A checkpoint without a results JSON is an INTERRUPTED cell, not a finished one.
# run_grid will retrain it on the next pass -- this is just so you can see it.
stems = {os.path.basename(p).replace('.pt', '') for p in ckpt}
done = {os.path.basename(p).replace('_results.json', '') for p in res}
print('interrupted :', len(stems) - len(stems & done))

!python -m src.model.run_grid --preflight 2>&1 | tail -3
